Logic:
1. Each check gives +-1
2. Error is one of the check's 4 neighbors
3. Compare the neighbors to confirm/pinpoint the exact error location
4. Apply X/Z to fix (code below tests to see whether the guessed fix would work)

In [ ]:
import random #Rand errors
from collections import deque #List that is fast to take items off the front of (used by BFS)

#Building grid/graph
#Data qubits sit in a d x d square. The qubit in row r, column c gets
#Number r(row)*d(num qubits in each row) + c(col) == converts (row,col) into a single qubit num
#For d = 3 (3x3 matrix) the numbers look like this:
#     0 1 2
#     3 4 5
#     6 7 8
#dxd grid acts together as one single big qubit (1 logical qubit), built out of tiny physical qubits
#Skip r full rows (each with d qubits) then go c cols in curr row
#A "check" is an alarm that watches a small group of qubits. Between the
#qubits there is a checkerboard of alarms: Z-type and X-type alternate.
#Z-type alarms catch X errors
#X-type alarms catch Z errors
#^^X and Z anticommute, so the opposite can detect the opposite error, not its own
#blueprint--explaining what each alarm can detect
def make_code(d): #d = side length of the grid (odd: 3, 5, 7, ...)
    z_type, x_type = [], [] #Two empty lists, one for each kind of alarm (Z, X)
    for r in range(-1, d): #Go through every row position, -1 means "just above the grid"
        for c in range(-1, d): #Go through every column position, top-to-bottom, left-to-right
            #Ex: (r, c) is the top-left corner of a little square of 4 qubits.
            qubits = [row * d + col #Turn a (row, column) spot into a qubit number
                      for row in (r, r + 1) #Square covers rows r and r+1
                      for col in (c, c + 1) #^^columns c and c+1
                      if 0 <= row < d and 0 <= col < d] #Throw away corners that fall off the grid
            is_z = (r + c) % 2 == 0 #Checkerboard: even == Z-type, odd == X-type (arbitrary choice)
            #Squares that touch along a side are opposite types, and they share 2 qubits (if they were the same they would cancel)
            #Alarms of opposite types must share an even number of qubits or they would disturb each other when measured
            #Squares that touch only a corner are the same type and share 1 qubit
            inside = 0 <= r <= d - 2 and 0 <= c <= d - 2 #Full plaquette (square) sitting inside the grid
            top_or_bottom = r in (-1, d - 1) and 0 <= c <= d - 2 #Half plaquette (square) on the top or bottom edge
            left_or_right = c in (-1, d - 1) and 0 <= r <= d - 2 #Half plaquette (square) on the left or right edge
            if inside or (top_or_bottom and is_z) or (left_or_right and not is_z):
                #Keep it if it's inside, or an edge piece of the right type
                #Z-type on top/bottom, X-type on left/right
                #Filtering to make stabilizers commute while encoding one logical qubit
                (z_type if is_z else x_type).append(qubits) #Track this alarm in the corresponding list
    return { #Return everything in one bundle
        "d": d, #Grid size
        "checks_for_x_errors": z_type, #Z-type alarms are the ones that notice X errors
        "checks_for_z_errors": x_type, #X-type alarms are the ones that notice Z errors
        #Below are for grading, decoder never sees them
        "logical_line_for_x_errors": [r * d for r in range(d)], #Left col of qubits
        "logical_line_for_z_errors": list(range(d)), #Top row of qubits
    }


#Make errors and flag alarms (alarms == an odd number of errors happened to the qubits you watch)
#Simulate errors
def make_random_errors(n, p): #n = number of qubits, p = chance each one breaks
    return [1 if random.random() < p else 0 #random.random() gives a number between 0 and 1; below p --> error (1)
            for _ in range(n)] #Do this once for every qubit
#Alarm goes off for 1,3,etc.(odd num) of broken qubits (errors cancel)
def get_syndrome(errors, checks): #Errors = list of 0/1 per qubit, checks = list of alarms
    return [sum(errors[q] for q in check) % 2 #Count broken qubits in this alarm's group, keep only odd/even
            for check in checks] #Do this for every alarm; result: 1 = rings, 0 = quiet


#Turn surface code into a reg graph and give the decoder a map to find the shortest chain of broken qubits between ringing alarms
#Build map/graph--alarms (stabilizer checks) become dots, qubits become lines, extra boundary dot
#Visual sense--Qubit touches at most 2 alarms, alarm touches 2/4 qubits
def build_graph(checks, n): #Checks = the alarms, n = number of qubits
    graph = {i: [] for i in range(len(checks))} #One dot per alarm, each starting with no connections
    graph["B"] = [] #Extra dot called "B" = the edge of the grid (boundary) cuz every qubit needs two ends
    for q in range(n): #Look at every qubit (every line on the map)
        touching = [i for i, check in enumerate(checks) #Find which alarms
                    if q in check] #watch this qubit
        if len(touching) == 2: #If two alarms watch it
            a, b = touching[0], touching[1] #The line connects those two alarms
        else: #If only one alarm watches it (it's on the edge)
            a, b = touching[0], "B" #Line connects that alarm to the boundary dot
        graph[a].append((b, q)) #At dot a, note "you can reach b using qubit q"
        graph[b].append((a, q)) #At dot b, note "you can reach a using qubit q"
    return graph #Finished map
#Decoder uses to find distances between paths
def find_path(graph, start, goal): #Shortest route from one dot to another (BFS)
    queue = deque([(start, [])])
    visited = {start} #Dots we've already seen, so we never go in circles
    while queue: #Keep going while there are places left to explore
        node, path = queue.popleft() #Take the oldest item (explores one step away, then two, then three...)
        if node == goal:
            return path #Shortest route
        for neighbor, qubit in graph[node]: #Look at every dot connected to this one
            if neighbor == "B" and goal != "B": #Boundary
                continue
            if neighbor not in visited: #Only visit dots we haven't seen yet
                visited.add(neighbor) #Mark it as seen
                queue.append((neighbor, path + [qubit])) #Remembering the qubit we crossed
    return None #No route exists


#Decoder guessing which qubit broke -- check nearest neighbor approach
#Looks at all ringing alarms and pairs them up
def decode(syndrome, checks, n): #Syndrome = which alarms rang (Only thing decoder knows)
    graph = build_graph(checks, n) #Draw the map
    correction = [0] * n #Guess at start: "no qubits broken"
    unpaired = [i for i, s in enumerate(syndrome) #Instructions: every alarm that rang,
                if s == 1] #in order (top-to-bottom, left-to-right)

    while unpaired: #keep going until every ringing alarm is examined
        current = unpaired.pop(0) #Take the first ringing alarm off the list

        options = [] #Every possible way to explain this alarm
        for other in unpaired: #option: pair it with another ringing alarm
            options.append((other, find_path(graph, current, other))) #remember the shortest route to it
        options.append(("B", find_path(graph, current, "B"))) #option: pair it with the boundary

        partner, path = min(options, key=lambda opt: len(opt[1]))
        #Choose the option with the fewest qubits on its route (fewer errors = more likely?);
        #If two options tie, min keeps the first one listed

        for q in path: #For every qubit along the chosen route
            correction[q] ^= 1 #Flip our guess (0->1, 1->0, so doing it twice cancels)
        if partner != "B": #If we paired with a real alarm (not the boundary)
            unpaired.remove(partner)
    return correction #Final guess: 1 = "flip this qubit back"
    #^^grabs unpaired[0] and pairs it with its closest neighbor right away == fast and simple


#Drawing visual images of process happening
#Combine real errors with the decoder's fix
def add_mod2(a, b): #Add two lists position by position
    return [(x + y) % 2 for x, y in zip(a, b)] #1+1 = 0, so an error and a matching fix cancel out
#Check if any remaining errors form a chain across grid (closed loops are fine)
def is_logical_error(leftover, line): #Did the stored information get flipped?
#Think: 000 --> 010 == 000 | 000 --> 111 == 111
    return sum(leftover[q] for q in line) % 2 == 1 #Count leftover errors on the grading line; odd = yes

def draw(x_errors, z_errors, d): #Print the grid as text
    symbols = {(0, 0): ".", (1, 0): "X", (0, 1): "Z", (1, 1): "Y"} #X?, Z? --> symbol; both = Y
    for r in range(d): #One printed line per grid row
        row = [] #Symbols for this row
        for c in range(d): #Go across the row
            q = r * d + c #Qubit number at this spot
            row.append(symbols[(x_errors[q], z_errors[q])]) #Look up the symbol for this qubit
        print("   " + " ".join(row)) #Print the row

def run_trial(code, x_errors, z_errors, verbose=True): #One full experiment; returns True if the decoder FAILED
    d = code["d"] #Grid side length
    n = d * d #Total num of qubits
    x_syn = get_syndrome(x_errors, code["checks_for_x_errors"]) #Which flags present because of X errors
    z_syn = get_syndrome(z_errors, code["checks_for_z_errors"]) #Which flags present because of Z errors
    x_fix = decode(x_syn, code["checks_for_x_errors"], n) #Decoder's guess for the X errors (sees only x_syn)
    z_fix = decode(z_syn, code["checks_for_z_errors"], n) #Decoder's guess for the Z errors (sees only z_syn)
    x_left = add_mod2(x_errors, x_fix) #X errors still left after applying the guess
    z_left = add_mod2(z_errors, z_fix) #Z errors still left after applying the guess
    failed = (is_logical_error(x_left, code["logical_line_for_x_errors"]) #Did X leftovers flip the stored info?
              or is_logical_error(z_left, code["logical_line_for_z_errors"])) #or Z leftovers?
    if verbose: #Print visual pics
        print("BEFORE (true errors):");   draw(x_errors, z_errors, d) #What really broke
        print("DECODER'S FIX:");          draw(x_fix, z_fix, d) #What the decoder flipped
        print("AFTER (errors + fix):");   draw(x_left, z_left, d) #What is left over (combo of errors and fix)
        print("FAILED (logical error)\n" if failed else "Passed\n") #Result
    return failed #True = the decoder failed

def logical_error_rate(code, p, trials=2000): #Run many random experiments and count failures
    failures = 0 #Start the count at zero
    n = code["d"] ** 2 #Num of qubits
    for _ in range(trials): #Repeat this many times
        x_errors = make_random_errors(n, p) #Rand X errors
        z_errors = make_random_errors(n, p) #Rand Z errors
        failures += run_trial(code, x_errors, z_errors, verbose=False) #Add 1 if the decoder failed to total failures
    return failures / trials #Ratio of trials that failed


#Testing algorithm with a simulation
SIZES = [3, 5, 7, 9] #Grid sizes to compare
ERROR_RATES = [0.005, 0.01, 0.02, 0.03, 0.04] #Chance of an error on each qubit
TRIALS = 4000 #Test trials, more trials = more helpful numbers but slower runtime overall?

if __name__ == "__main__": #Only run test below when you run this file directly
    #random.seed(0) #Uncommented to get the same random numbers every run
    code5 = make_code(5) #Build a 5x5 grid (for ex)
    run_trial(code5, make_random_errors(25, 0.08), make_random_errors(25, 0.08)) #Draw one random example

    codes = {d: make_code(d) for d in SIZES} #Build each grid once
    print("p       " + "   ".join(f"d={d}  " for d in SIZES)) #Table header
    for p in ERROR_RATES: #One table row per error rate
        row = [logical_error_rate(codes[d], p, TRIALS) for d in SIZES] #Failure rate for each grid size
        print(f"{p:<7} " + "  ".join(f"{r:.4f}" for r in row)) #Print the row